# Customer Catalogue GP Homologation

## Step 1 - Load, validate and normalize Global Parents

This notebook:

- Loads the customer catalogue
- Validates required columns
- Counts unique Ship To Numbers per Global Parent
- Creates a normalized GP name
- Exports a summary for review

The original catalogue is NEVER modified.

In [121]:
import pandas as pd
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process
from rapidfuzz import fuzz

In [122]:
# ==========================================
# Configuration
# ==========================================

INPUT_FILE = 'Industrial_Market.csv'

OUTPUT_FOLDER = "output"

GP_COLUMN = "Global Parent"

SHIP_TO_COLUMN = "Ship To Number"

REMOVE_ACCENTS = True
REMOVE_PUNCTUATION = True
REMOVE_EXTRA_SPACES = True
REMOVE_LEGAL_SUFFIXES = True

In [123]:
LEGAL_SUFFIXES = [

    "INCORPORATED",
    "INC",
    "CORPORATION",
    "CORP",
    "COMPANY",
    "CO",
    "LIMITED",
    "LTD",
    "LLC",
    "LLP",
    "PLC",
    "LP",
    "GMBH",
    "AG",
    "BV",
    "NV",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "PTY",
    "PVT",
    "KK"

]

In [124]:
def normalize_gp(name):

    if pd.isna(name):
        return ""

    text = str(name).upper()

    if REMOVE_ACCENTS:
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

    if REMOVE_PUNCTUATION:
        text = re.sub(r"[^\w\s]", " ", text)

    if REMOVE_LEGAL_SUFFIXES:

        words = text.split()

        words = [
            word
            for word in words
            if word not in LEGAL_SUFFIXES
        ]

        text = " ".join(words)

    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [125]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")

Rows loaded: 49,492


In [ ]:
required_columns = [
    GP_COLUMN,
    SHIP_TO_COLUMN
]

missing = [

    column
    for column in required_columns
    if column not in df.columns

]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns found.")

All required columns found.


In [127]:
df["Normalized GP"] = df[GP_COLUMN].apply(normalize_gp)

df["Normalization Changed"] = (
    df[GP_COLUMN].str.upper().fillna("")
    !=
    df["Normalized GP"]
)

In [128]:
gp_summary = (

    df.groupby(GP_COLUMN)
      .agg(

          Ship_To_Count=(
              SHIP_TO_COLUMN,
              "nunique"
          ),

          Row_Count=(
              SHIP_TO_COLUMN,
              "count"
          ),

          GP_Country=(
              "GP Country",
              "first"
          ),

          Normalized_GP=(
              "Normalized GP",
              "first"
          ),

          Normalization_Changed=(
              "Normalization Changed",
              "first")

      )

      .reset_index()

)

In [129]:
gp_summary = gp_summary.sort_values(

    by=[
        "Ship_To_Count",
        GP_COLUMN
    ],

    ascending=[
        False,
        True
    ]

)

In [131]:
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

output_file = Path(OUTPUT_FOLDER) / "01_gp_summary.csv"

gp_summary.to_csv(

    output_file,

    index=False

)

print(f"Saved to: {output_file}")

Saved to: output\01_gp_summary.csv


In [132]:
exact_groups = (
    gp_summary
    .groupby("Normalized_GP")
    .agg(
        Variants=(GP_COLUMN, list),
        Number_of_Variants=(GP_COLUMN, "nunique"),
        Total_Ship_To_Count=("Ship_To_Count", "sum"),
        Total_Row_Count=("Row_Count", "sum"),
        Countries=("GP_Country", lambda x: sorted(set(x.dropna()))),
        Any_Normalization_Changed=("Normalization_Changed", "any")
    )
    .reset_index()
)

In [133]:
exact_groups = exact_groups[
    exact_groups["Number_of_Variants"] > 1
].copy()

In [134]:
exact_groups = exact_groups.sort_values(
    by=[
        "Total_Ship_To_Count",
        "Number_of_Variants"
    ],
    ascending=False
)

In [136]:
output_file = Path(OUTPUT_FOLDER) / "02_exact_normalized_groups.csv"

exact_groups.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\02_exact_normalized_groups.csv


In [137]:
entities = exact_groups.copy()

entities = entities.rename(
    columns={
        "Normalized_GP": "Representative",
        "Variants": "Members",
        "Total_Ship_To_Count": "Ship_To_Count",
        "Total_Row_Count": "Row_Count",
        "Countries": "Countries"
    }
)

In [138]:
entities.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(1, len(entities) + 1)
    ]
)

In [139]:
grouped_gps = set()

for members in entities["Members"]:
    grouped_gps.update(members)

singletons = gp_summary[
    ~gp_summary[GP_COLUMN].isin(grouped_gps)
].copy()

In [140]:
singletons["Representative"] = singletons["Normalized_GP"]

singletons["Members"] = singletons[GP_COLUMN].apply(lambda x: [x])

singletons["Countries"] = singletons["GP_Country"].apply(
    lambda x: [] if pd.isna(x) else [x]
)


start = len(entities) + 1

singletons.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(start, start + len(singletons))
    ]
)

In [141]:
singletons = singletons[
    [
        "Entity_ID",
        "Representative",
        "Members",
        "Ship_To_Count",
        "Row_Count",
        "Countries"
    ]
]

In [142]:
entities = pd.concat(
    [
        entities,
        singletons
    ],
    ignore_index=True
)

In [143]:
entities = entities.sort_values(
    by="Ship_To_Count",
    ascending=False
)

In [145]:
entities_export = entities.copy()


def list_to_string(value):

    if isinstance(value, list):
        return " | ".join(value)

    if pd.isna(value):
        return ""

    return str(value)


entities_export["Members"] = entities_export["Members"].apply(list_to_string)

entities_export["Countries"] = entities_export["Countries"].apply(list_to_string)


output_file = Path(OUTPUT_FOLDER) / "03_entities.csv"

entities_export.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\03_entities.csv


In [146]:
# ===================================
# Fuzzy Matching Configuration
# ===================================

TOP_CANDIDATES = 10

MIN_SCORE = 95

SCORER = fuzz.token_sort_ratio

In [147]:
entity_names = entities["Representative"].tolist()

In [148]:
def generate_candidates(name):

    return process.extract(

        query=name,
        choices=entity_names,
        scorer=SCORER,
        limit=TOP_CANDIDATES

    )

In [149]:
def calculate_scores(name1, name2):

    return {

        "Ratio": round(fuzz.ratio(name1, name2), 1),

        "Partial_Ratio": round(fuzz.partial_ratio(name1, name2), 1),

        "Token_Sort": round(fuzz.token_sort_ratio(name1, name2), 1),

        "Token_Set": round(fuzz.token_set_ratio(name1, name2), 1)

    }

In [150]:
def classify_match(scores):

    if scores["Token_Set"] >= 98 and scores["Ratio"] >= 85:
        return "High"

    if scores["Token_Set"] >= 95 and scores["Ratio"] >= 70:
        return "Medium"

    return "Low"


def is_high_confidence(scores):

    return classify_match(scores) != "Low"

In [151]:
entity_lookup = (
    entities
    .set_index("Representative")
    .to_dict("index")
)

In [152]:
fuzzy_matches = []

In [153]:
for _, entity in entities.iterrows():

    representative = entity["Representative"]

    candidates = generate_candidates(representative)

    for candidate_name, _, _ in candidates:

        # Ignore itself
        if candidate_name == representative:
            continue

        scores = calculate_scores(
            representative,
            candidate_name
        )

        if not is_high_confidence(scores):
            continue

        candidate = entity_lookup.get(candidate_name)

        if candidate is None:
            continue

        confidence = classify_match(scores)

        fuzzy_matches.append({

            "Entity_1": entity["Entity_ID"],
            "Representative_1": representative,
            "Members_1": entity["Members"],
            "Countries_1": entity["Countries"],
            "Ship_To_1": entity["Ship_To_Count"],

            "Entity_2": candidate["Entity_ID"],
            "Representative_2": candidate_name,
            "Members_2": candidate["Members"],
            "Countries_2": candidate["Countries"],
            "Ship_To_2": candidate["Ship_To_Count"],

            "Confidence": confidence,
            "Combined_Name_Length": len(representative) + len(candidate_name),

            **scores

})
        
        

In [154]:
fuzzy_matches = pd.DataFrame(fuzzy_matches)

In [155]:
fuzzy_matches["Pair"] = fuzzy_matches.apply(

    lambda row: tuple(sorted([
        row["Entity_1"],
        row["Entity_2"]
    ])),

    axis=1
)

fuzzy_matches = (
    fuzzy_matches
    .drop_duplicates("Pair")
    .drop(columns="Pair")
)

In [156]:
fuzzy_matches["Total_Ship_To"] = (
    fuzzy_matches["Ship_To_1"] +
    fuzzy_matches["Ship_To_2"]
)

confidence_order = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}

fuzzy_matches["Confidence_Order"] = (
    fuzzy_matches["Confidence"]
    .map(confidence_order)
)

fuzzy_matches = fuzzy_matches.sort_values(

    by=[
        "Confidence_Order",
        "Total_Ship_To",
        "Token_Set",
        "Ratio"
    ],

    ascending=False

).drop(columns="Confidence_Order")

In [158]:
fuzzy_matches_export = fuzzy_matches[
    fuzzy_matches["Confidence"] != "Low"
].copy()

fuzzy_matches_export["Members_1"] = fuzzy_matches_export["Members_1"].apply(list_to_string)
fuzzy_matches_export["Members_2"] = fuzzy_matches_export["Members_2"].apply(list_to_string)

fuzzy_matches_export["Countries_1"] = fuzzy_matches_export["Countries_1"].apply(list_to_string)
fuzzy_matches_export["Countries_2"] = fuzzy_matches_export["Countries_2"].apply(list_to_string)


output_file = Path(OUTPUT_FOLDER) / "04_fuzzy_matches.csv"

fuzzy_matches_export.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\04_fuzzy_matches.csv
